### Calculate Metrics (KGE, NSE, PBIAS)

For multiple experimental runs, you can process each input CSV and generate corresponding output CSVs with the Issue flag:

In [13]:
#!/usr/bin/env python
"""
Hydrology Model Evaluation Script v2
- Multiple station filters from input file
- Consistent stations across all plots
- Optimized, vectorized, cached
"""
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any, Set
from itertools import cycle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# METRIC DESCRIPTIONS
# =============================================================================
METRIC_DESCRIPTIONS = {
    'KGE': {
        'Name': 'Kling-Gupta Efficiency',
        'Description': 'Combines correlation, bias, and variability ratios.',
        'Range': '(-∞, 1], 1 = perfect',
        'Units': 'Dimensionless',
        'Interpretation': 'Higher → better; <0 worse than mean.'
    },
    'NSE': {
        'Name': 'Nash-Sutcliffe Efficiency',
        'Description': 'Variance explained relative to observed mean.',
        'Range': '(-∞, 1], 1 = perfect',
        'Units': 'Dimensionless',
        'Interpretation': 'Higher → better.'
    },
    'PBIAS': {
        'Name': 'Percent Bias',
        'Description': 'Average tendency to over/under-predict.',
        'Range': '(-∞, ∞), 0 = no bias',
        'Units': '%',
        'Interpretation': 'Closer to 0 → better.'
    },
    'RMSE': {
        'Name': 'Root Mean Square Error',
        'Description': 'Average magnitude of errors (sensitive to outliers).',
        'Range': '[0, ∞)',
        'Units': 'Same as data',
        'Interpretation': 'Lower → better.'
    },
    'MAE': {
        'Name': 'Mean Absolute Error',
        'Description': 'Average absolute error (robust to outliers).',
        'Range': '[0, ∞)',
        'Units': 'Same as data',
        'Interpretation': 'Lower → better.'
    },
    'R2': {
        'Name': 'Coefficient of Determination',
        'Description': 'Proportion of variance explained.',
        'Range': '[0, 1]',
        'Units': 'Dimensionless',
        'Interpretation': 'Higher → better.'
    },
    'MAPE': {
        'Name': 'Mean Absolute Percentage Error',
        'Description': 'Relative error as percentage (avoid division by zero).',
        'Range': '[0, ∞)',
        'Units': '%',
        'Interpretation': 'Lower → better.'
    },
    'VE': {
        'Name': 'Volume Error',
        'Description': 'Total volumetric bias as percentage.',
        'Range': '(-∞, ∞)',
        'Units': '%',
        'Interpretation': 'Closer to 0 → better.'
    }
}

# =============================================================================
# UNIFIED VECTORIZED METRIC CALCULATION
# =============================================================================
def compute_metrics_vectorized(
    sim: pd.Series, obs: pd.Series
) -> Tuple[Dict[str, float], Optional[str]]:
    """
    Compute all metrics in one pass using pandas vectorized operations.
    Returns (metrics_dict, issue_message or None)
    """
    df = pd.DataFrame({'sim': sim, 'obs': obs}).dropna()
    if len(df) < 2:
        return {m: np.nan for m in METRIC_DESCRIPTIONS}, "Insufficient valid data points"
    sim, obs = df['sim'], df['obs']
    metrics = {}
    issue = None
    # Basic stats
    mean_sim, mean_obs = sim.mean(), obs.mean()
    std_sim, std_obs = sim.std(ddof=1), obs.std(ddof=1)
    sum_obs = obs.sum()
    # KGE
    if mean_obs == 0 or std_obs == 0 or std_sim == 0:
        metrics['KGE'] = np.nan
        issue = issue or "Zero mean/std in KGE"
    else:
        r = np.corrcoef(sim, obs)[0, 1]
        if np.isnan(r):
            metrics['KGE'] = np.nan
            issue = issue or "Invalid correlation"
        else:
            alpha = std_sim / std_obs
            beta = mean_sim / mean_obs
            metrics['KGE'] = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    # NSE
    if std_obs == 0:
        metrics['NSE'] = np.nan
        issue = issue or "Zero std in NSE"
    else:
        metrics['NSE'] = 1 - ((obs - sim) ** 2).sum() / ((obs - mean_obs) ** 2).sum()
    # PBIAS & VE (same)
    if sum_obs == 0:
        metrics['PBIAS'] = metrics['VE'] = np.nan
        issue = issue or "Sum of observed is zero"
    else:
        bias = (sim.sum() - sum_obs) * 100 / sum_obs
        metrics['PBIAS'] = metrics['VE'] = bias
    # RMSE & MAE
    metrics['RMSE'] = np.sqrt(((obs - sim) ** 2).mean())
    metrics['MAE'] = (obs - sim).abs().mean()
    # R2
    if std_obs == 0 or std_sim == 0:
        metrics['R2'] = np.nan
        issue = issue or "Zero std in R2"
    else:
        r = np.corrcoef(sim, obs)[0, 1]
        metrics['R2'] = r ** 2 if not np.isnan(r) else np.nan
    # MAPE (avoid div by zero)
    nonzero = obs != 0
    if nonzero.sum() < 2:
        metrics['MAPE'] = np.nan
        issue = issue or "Insufficient non-zero obs for MAPE"
    else:
        metrics['MAPE'] = ((obs[nonzero] - sim[nonzero]).abs() / obs[nonzero].abs()).mean() * 100
    return metrics, issue

# =============================================================================
# CORE PROCESSING
# =============================================================================
def compute_efficiency_custom(
    df: pd.DataFrame,
    prefix_obs: str = 'QOMEAS_',
    prefix_simu: str = 'QOSIM_'
) -> pd.DataFrame:
    """Compute all metrics for each station efficiently."""
    obs_cols = [c for c in df.columns if c.startswith(prefix_obs)]
    results = []
    for col_obs in obs_cols:
        station_id = col_obs[len(prefix_obs):].strip()
        col_sim = f"{prefix_simu}{station_id}"
        if col_sim not in df.columns:
            results.append({
                'StationID': station_id,
                **{m: np.nan for m in METRIC_DESCRIPTIONS},
                'Issue': True,
                'IssueMessage': 'Missing simulated column'
            })
            continue
        metrics, issue_msg = compute_metrics_vectorized(df[col_sim], df[col_obs])
        metrics.update({
            'StationID': station_id,
            'Issue': issue_msg is not None,
            'IssueMessage': issue_msg or ''
        })
        results.append(metrics)
    return pd.DataFrame(results).sort_values('StationID').reset_index(drop=True)

def save_metric_descriptions(output_dir: Path, filename: str = 'metrics_description.txt'):
    """Save metric descriptions to file."""
    path = output_dir / filename
    with path.open('w', encoding='utf-8') as f:
        f.write("Hydrology Metrics Description\n" + "="*30 + "\n\n")
        for metric, info in METRIC_DESCRIPTIONS.items():
            f.write(f"Metric: {info['Name']} ({metric})\n")
            for key in ['Description', 'Range', 'Units', 'Interpretation']:
                f.write(f"{key}: {info[key]}\n")
            f.write("-" * 30 + "\n\n")
    print(f"Metric descriptions saved to '{path.name}'")

def process_flow_csv(
    input_csv: Path,
    output_csv: Path,
    prefix_obs: str = 'QOMEAS_',
    prefix_simu: str = 'QOSIM_',
    skip_days: int = 0,
    missing_value: Any = None
):
    """Process one CSV file."""
    if not input_csv.exists():
        raise FileNotFoundError(f"Input not found: {input_csv}")
    df = pd.read_csv(input_csv)
    if missing_value is not None:
        df = df.replace(missing_value, np.nan)
    if skip_days > 0:
        if skip_days >= len(df):
            raise ValueError(f"skip_days ({skip_days}) exceeds rows ({len(df)})")
        df = df.iloc[skip_days:].reset_index(drop=True)
        print(f"Skipped {skip_days} days from {input_csv.name}")
    results_df = compute_efficiency_custom(df, prefix_obs, prefix_simu)
    results_df.to_csv(output_csv, index=False)
    print(f"Metrics saved to {output_csv.name}")
    save_metric_descriptions(output_csv.parent)
    issues = results_df['Issue'].sum()
    if issues:
        print(f"{issues} stations had issues.")

def process_all_runs(
    base_path: Path | str,
    mesh_versions: List[str],
    gru_types: List[str],
    skip_days: int = 10,
    missing_value: Any = None
) -> List[Path]:
    """Process all runs and return output CSV paths."""
    base_path = Path(base_path)
    output_csvs = []
    for mesh in mesh_versions:
        for gru in gru_types:
            gru_path = base_path / mesh / gru
            if not gru_path.is_dir():
                print(f"Skipping missing: {gru_path}")
                continue
            for sub in gru_path.iterdir():
                if not sub.is_dir():
                    continue
                input_csv = sub / "MESH_output_streamflow.csv"
                output_csv = sub / f"metrics_{mesh}_{gru}_{sub.name}.csv"
                if not input_csv.exists():
                    print(f"Missing input: {input_csv}")
                    continue
                print(f"Processing: {mesh}/{gru}/{sub.name}")
                process_flow_csv(input_csv, output_csv, skip_days=skip_days, missing_value=missing_value)
                output_csvs.append(output_csv)
    return output_csvs

# =============================================================================
# STATION FILTERING: MULTIPLE CONDITIONS
# =============================================================================
FilterCondition = Tuple[str, str]  # (column, condition_expr)

def apply_station_filters(
    gdf: gpd.GeoDataFrame,
    filters: List[FilterCondition]
) -> gpd.GeoDataFrame:
    """Apply multiple filter conditions safely using pandas query syntax."""
    if not filters:
        return gdf.copy()

    filtered = gdf.copy()
    applied = []
    for col, cond in filters:
        if col not in filtered.columns:
            raise ValueError(f"Filter column '{col}' not in stations file. Available: {list(filtered.columns)}")
        try:
            # Use backticks for column names with spaces/special chars
            col_escaped = f"`{col}`" if ' ' in col or any(c in col for c in ".$[]") else col
            query_expr = f"{col_escaped} {cond}"
            before = len(filtered)
            filtered = filtered.query(query_expr)
            applied.append(f"{col} {cond}")
            print(f"  Filter: {col} {cond} → {len(filtered)} (from {before})")
        except Exception as e:
            raise ValueError(f"Invalid filter condition: `{col} {cond}` → {e}")
    
    print(f"Total filters applied ({len(filters)}): {', '.join(applied)}")
    print(f"Final stations: {len(filtered)} / {len(gdf)}")
    return filtered

def load_stations_with_filters(
    stations_file: Path | str,
    station_id_col: str,
    filters: Optional[List[FilterCondition]] = None
) -> pd.DataFrame:
    """Load stations and apply multiple filters."""
    stations_file = Path(stations_file)
    if not stations_file.exists():
        raise FileNotFoundError(f"Stations file not found: {stations_file}")
    
    gdf = gpd.read_file(stations_file)
    if station_id_col not in gdf.columns:
        raise ValueError(f"'{station_id_col}' not in columns: {list(gdf.columns)}")
    
    gdf = gdf.to_crs(epsg=4326)
    filtered_gdf = apply_station_filters(gdf, filters or [])
    
    return pd.DataFrame({
        'StationID': filtered_gdf[station_id_col],
        'Longitude': filtered_gdf.geometry.x,
        'Latitude': filtered_gdf.geometry.y
    }).reset_index(drop=True)

# =============================================================================
# METRICS CACHE
# =============================================================================
class MetricsCache:
    """Cache metric CSVs and provide filtered views."""
    def __init__(self, csv_files: List[Path]):
        self.csv_files = [Path(p) for p in csv_files]
        self.dfs: List[pd.DataFrame] = []
        self.run_names: List[str] = []
        self._load_all()

    def _load_all(self):
        print(f"Loading {len(self.csv_files)} metric files...")
        for csv in self.csv_files:
            df = pd.read_csv(csv)
            if 'StationID' not in df.columns:
                print(f"Skipping invalid: {csv.name} (no StationID)")
                continue
            name = csv.stem.replace('metrics_', '').replace('_', ' ').title()
            self.run_names.append(name)
            self.dfs.append(df)
        print(f"Loaded {len(self.dfs)} valid runs.")

    def get_metric_data(self, metric: str, allowed_stations: Optional[Set[str]] = None) -> Tuple[List[pd.DataFrame], Set[str], List[str]]:
        """Return filtered dataframes, common valid stations, and names."""
        if metric not in METRIC_DESCRIPTIONS:
            raise ValueError(f"Invalid metric: {metric}")
        
        valid_sets = []
        filtered_dfs = []

        for df in self.dfs:
            if metric not in df.columns:
                continue
            mask = df['StationID'].isin(allowed_stations) if allowed_stations else slice(None)
            filtered = df.loc[mask].copy()
            valid = set(filtered[filtered[metric].notna()]['StationID'])
            valid_sets.append(valid)
            filtered_dfs.append(filtered)

        if not valid_sets:
            raise ValueError(f"No data for metric '{metric}'")

        common = set.intersection(*valid_sets)
        if allowed_stations:
            common &= allowed_stations

        # Report excluded
        all_stations = set()
        for df in self.dfs:
            all_stations.update(df['StationID'])
        excluded = all_stations - common
        if excluded:
            excl_file = self.csv_files[0].parent / f"excluded_stations_{metric.lower()}.txt"
            excl_file.write_text("\n".join(sorted(excluded)), encoding='utf-8')
            print(f"Excluded {len(excluded)} stations → {excl_file.name}")

        return filtered_dfs, common, self.run_names

# =============================================================================
# PLOTTING: CDF
# =============================================================================
def plot_cumulative_distribution(
    cache: MetricsCache,
    metric: str = 'KGE',
    output_file: str | Path = 'cdf_plot.png',
    common_stations: Optional[Set[str]] = None
):
    output_file = Path(output_file)
    dfs, common, names = cache.get_metric_data(metric, common_stations)
    common = common_stations or common

    plt.figure(figsize=(10 + len(dfs)//3, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    styles = cycle(['-', '--', '-.', ':'])

    all_vals = []
    for i, (df, name) in enumerate(zip(dfs, names)):
        data = df[df['StationID'].isin(common)][metric].dropna()
        if data.empty:
            continue
        sorted_val = np.sort(data)
        y = np.linspace(0, 1, len(sorted_val))
        plt.plot(sorted_val, y, label=name, color=colors[i % 10], linestyle=next(styles), linewidth=2)
        all_vals.extend(data)

    if all_vals:
        mn, mx = min(all_vals), max(all_vals)
        r = mx - mn
        m = 0.02 * r if r > 0 else 0.02
        plt.xlim(mn - m, mx + m)

    unit = "(%)" if metric in ["PBIAS","MAPE","VE"] else "(units)" if metric in ["RMSE","MAE"] else ""
    plt.title(f'CDF of {metric} ({len(common)} Stations)')
    plt.xlabel(f'{metric} {unit}')
    plt.ylabel('Cumulative Probability')
    plt.grid(True, ls='--', alpha=0.7)
    plt.legend(title='Runs', loc='upper left')
    plt.tight_layout()
    plt.savefig(output_file, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"CDF saved: {output_file.name}")

# =============================================================================
# PLOTTING: SPATIAL SINGLE RUN
# =============================================================================
def xxplot_spatial_stations(
    cache: MetricsCache,
    coords: pd.DataFrame,
    metric: str = 'KGE',
    run_index: int = 0,
    output_file: str | Path = 'spatial_plot.png',
    common_stations: Optional[Set[str]] = None
):
    output_file = Path(output_file)
    run_index = min(run_index, len(cache.dfs) - 1)
    df = cache.dfs[run_index]
    name = cache.run_names[run_index]

    plot_df = coords[coords['StationID'].isin(common_stations)].merge(
        df[['StationID', metric]], on='StationID'
    ).dropna(subset=[metric])

    if plot_df.empty:
        print(f"No valid data for spatial plot: {name}")
        return

    plt.figure(figsize=(10, 8))
    vmin, vmax = plot_df[metric].min(), plot_df[metric].max()
    r = vmax - vmin
    m = 0.02 * r if r > 0 else 0.02
    scatter = plt.scatter(plot_df['Longitude'], plot_df['Latitude'],
                          c=plot_df[metric], cmap='viridis', s=100,
                          edgecolor='k', linewidth=0.5, vmin=vmin-m, vmax=vmax+m)
    unit = "(%)" if metric in ["PBIAS","MAPE","VE"] else "(units)" if metric in ["RMSE","MAE"] else ""
    plt.colorbar(scatter, label=f'{metric} {unit}')
    plt.title(f'{metric}: {name}\n({len(plot_df)} Stations)')
    plt.xlabel('Longitude (°)')
    plt.ylabel('Latitude (°)')
    plt.grid(True, ls='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(output_file, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"Spatial plot saved: {output_file.name}")

# =============================================================================
# PLOTTING: SPATIAL SINGLE RUN
# =============================================================================
def plot_spatial_stations(
    cache: MetricsCache,
    coords: pd.DataFrame,
    metric: str = 'KGE',
    run_index: int = 0,
    output_file: str | Path = 'spatial_plot.png',
    common_stations: Optional[Set[str]] = None,
    basin_shapefile: Optional[str | Path] = None,     # ← NEW
    point_size: int = 100,                             # ← NEW
    basin_linewidth: float = 1.2                       # ← NEW
):
    import matplotlib.pyplot as plt
    import geopandas as gpd
    from pathlib import Path

    output_file = Path(output_file)
    run_index = min(run_index, len(cache.dfs) - 1)
    df = cache.dfs[run_index]
    name = cache.run_names[run_index]

    plot_df = coords[coords['StationID'].isin(common_stations)].merge(
        df[['StationID', metric]], on='StationID'
    ).dropna(subset=[metric])

    if plot_df.empty:
        print(f"No valid data for spatial plot: {name}")
        return

    plt.figure(figsize=(10, 8))
    ax = plt.gca()

    # === NEW: Basin boundary ===
    if basin_shapefile is not None:
        basin_path = Path(basin_shapefile)
        if basin_path.with_suffix('.shp').exists():
            try:
                basin = gpd.read_file(basin_path.with_suffix('.shp'))
                if basin.crs is not None and basin.crs.to_string() != "EPSG:4326":
                    basin = basin.to_crs("EPSG:4326")
                basin.boundary.plot(ax=ax, color='black', linewidth=basin_linewidth)
                basin.plot(ax=ax, facecolor='none', edgecolor='gray', linewidth=basin_linewidth*0.6, alpha=0.3)
            except Exception as e:
                print(f"Warning: Could not load basin shapefile: {e}")

    vmin, vmax = plot_df[metric].min(), plot_df[metric].max()
    r = vmax - vmin
    m = 0.02 * r if r > 0 else 0.02

    scatter = plt.scatter(plot_df['Longitude'], plot_df['Latitude'],
                          c=plot_df[metric], cmap='viridis', s=point_size,
                          edgecolor='k', linewidth=0.5, vmin=vmin-m, vmax=vmax+m)

    unit = "(%)" if metric in ["PBIAS","MAPE","VE"] else "(units)" if metric in ["RMSE","MAE"] else ""
    plt.colorbar(scatter, label=f'{metric} {unit}')
    plt.title(f'{metric}: {name}\n({len(plot_df)} Stations)')
    plt.xlabel('Longitude (°)')
    plt.ylabel('Latitude (°)')
    plt.grid(True, ls='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(output_file, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"Spatial plot saved: {output_file.name}")

# =============================================================================
# PLOTTING: SPATIAL COMPARISON
# =============================================================================
def plot_spatial_comparison(
    cache: MetricsCache,
    coords: pd.DataFrame,
    metric: str = 'KGE',
    run_index1: int = 0,
    run_index2: int = 1,
    output_file: str | Path = 'spatial_comparison_plot.png',
    common_stations: Optional[Set[str]] = None,
    basin_shapefile: Optional[str | Path] = None,      # ← NEW
    point_size: int = 100,                             # ← NEW
    basin_linewidth: float = 1.2                       # ← NEW
):
    import matplotlib.pyplot as plt
    import geopandas as gpd
    from pathlib import Path

    output_file = Path(output_file)
    if len(cache.dfs) < 2:
        print("Need at least 2 runs.")
        return

    i1, i2 = min(run_index1, len(cache.dfs)-1), min(run_index2, len(cache.dfs)-1)
    df1 = cache.dfs[i1][['StationID', metric]].rename(columns={metric: f'{metric}_1'})
    df2 = cache.dfs[i2][['StationID', metric]].rename(columns={metric: f'{metric}_2'})
    name1, name2 = cache.run_names[i1], cache.run_names[i2]

    merged = df1.merge(df2, on='StationID')
    merged = merged[merged['StationID'].isin(common_stations)]
    merged['Diff'] = merged[f'{metric}_1'] - merged[f'{metric}_2']
    plot_df = coords.merge(merged, on='StationID').dropna(subset=['Diff'])

    if plot_df.empty:
        print("No valid data for comparison.")
        return

    plt.figure(figsize=(10, 8))
    ax = plt.gca()

    # === NEW: Basin boundary with controllable linewidth ===
    if basin_shapefile is not None:
        basin_path = Path(basin_shapefile)
        if basin_path.with_suffix('.shp').exists():
            try:
                basin = gpd.read_file(basin_path.with_suffix('.shp'))
                if basin.crs is not None and basin.crs.to_string() != "EPSG:4326":
                    basin = basin.to_crs("EPSG:4326")
                basin.boundary.plot(ax=ax, color='black', linewidth=basin_linewidth)
                basin.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=basin_linewidth*0.6, alpha=0.3)
            except Exception as e:
                print(f"Warning: Could not load basin shapefile: {e}")

    diff = plot_df['Diff']
    bound = max(diff.abs().max(), 0.1)
    scatter = plt.scatter(plot_df['Longitude'], plot_df['Latitude'],
                          c=diff, cmap='RdBu', s=point_size, edgecolor='k', linewidth=0.5,
                          vmin=-bound, vmax=bound)
    unit = "(%)" if metric in ["PBIAS","MAPE","VE"] else "(units)" if metric in ["RMSE","MAE"] else ""
    plt.colorbar(scatter, label=f'{metric} Δ ({name1} - {name2}) {unit}')
    plt.title(f'{metric} Δ: {name1} vs {name2}\n({len(plot_df)} Stations)')
    plt.xlabel('Longitude (°)')
    plt.ylabel('Latitude (°)')
    plt.grid(True, ls='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(output_file, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"Comparison plot saved: {output_file.name}")

In [20]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    # ==============================
    # USER CONFIGURATION
    # ==============================
    base_path = Path(r"D:\Zelalem\RUNs")
    station_id_col = "Obs_NM"
    stations_file = Path(r"D:\Zelalem\WSC\combined_discharge_stations_comids2.gpkg")
    basin_file = Path(r"D:/Zelalem/WSC/CanTrans_MERIT_StudyDomain.shp")

    # === FILTERS: List of (column, condition) ===
    station_filters: List[FilterCondition] = [
         ("DA_Obs", "> 100"),               # ← km²
 #       ("DA_Obs", "<= 1000000"),         # ← km²
 #       ("RHBN", "== 'TRUE'"),            # ← This is best for boolean
 #       ("IsRegulate", "== False"),       # ← This is best for boolean
        ("PRecord", ">= 10"),       # ← This is best for boolean        
 #       ("COMID_CAMELS", "> 0"),           # ← CAMEL-SPAT stations
        ("DA_Diff", "> -10"),
        ("DA_Diff", "< 10"),
        ("HYD_STATUS", "== 'A'"),
        ("KGE_x", ">= -1"),
        ("KGE_y", ">= -1"),
        ("KGE", ">= -1")
        # Add more as needed
    ]

    mesh_versions = ["MESH_CaSRv2p1", "MESH_CaSRv3p1"]
    gru_types = ["Average_GRU_Params", "Distributed_GRU_Params"]
    skip_days = 365
    missing_value = -1

    # ==============================
    # PROCESS RUNS
    # ==============================
    print("Processing all model runs...")
    output_csvs = process_all_runs(
        base_path, mesh_versions, gru_types,
        skip_days=skip_days, missing_value=missing_value
    )

    if not output_csvs:
        raise RuntimeError("No output CSVs generated.")

    # ==============================
    # LOAD STATIONS WITH FILTERS
    # ==============================
    print("\nLoading and filtering stations...")
    coords_df = load_stations_with_filters(
        stations_file, station_id_col, filters=station_filters
    )
    common_stations = set(coords_df['StationID'])
    print(f"Using {len(common_stations)} stations for all plots.")

    # ==============================
    # CACHE METRICS
    # ==============================
    cache = MetricsCache(output_csvs)

    # ==============================
    # PLOT FOR EACH METRIC
    # ==============================
    metrics_to_plot = ['KGE', 'NSE', 'PBIAS']
    for metric in metrics_to_plot:
        suffix = f"filtered_{metric.lower()}"
        
        # CDF
        plot_cumulative_distribution(
            cache, metric=metric,
            output_file=f"cdf_{suffix}.png",
            common_stations=common_stations
        )
        
        # Spatial (first run)
        plot_spatial_stations(
            cache, coords_df, metric=metric,
            run_index=0,
            output_file=f"spatial_run0_{suffix}.png",
            common_stations=common_stations,
            basin_shapefile=basin_file,    # ← your basin polygon
            point_size=30,                 # ← adjust as needed
            basin_linewidth=0.15           # ← thicker boundary
        )
        
        # Comparison (if >1 run)
        if len(cache.dfs) > 1:
            plot_spatial_comparison(
                cache, coords_df, metric=metric,
                run_index1=0, run_index2=1,
                output_file=f"spatial_cmp_{suffix}.png",
                common_stations=common_stations,
                basin_shapefile=basin_file,    # ← your basin polygon
                point_size=30,                 # ← adjust as needed
                basin_linewidth=0.15           # ← thicker boundary
            )

    print("\nAll done!")

Processing all model runs...
Processing: MESH_CaSRv2p1/Average_GRU_Params/MESH_1p5p5
Skipped 365 days from MESH_output_streamflow.csv
Metrics saved to metrics_MESH_CaSRv2p1_Average_GRU_Params_MESH_1p5p5.csv
Metric descriptions saved to 'metrics_description.txt'
455 stations had issues.
Skipping missing: D:\Zelalem\RUNs\MESH_CaSRv2p1\Distributed_GRU_Params
Processing: MESH_CaSRv3p1/Average_GRU_Params/MESH_1p5p5
Skipped 365 days from MESH_output_streamflow.csv
Metrics saved to metrics_MESH_CaSRv3p1_Average_GRU_Params_MESH_1p5p5.csv
Metric descriptions saved to 'metrics_description.txt'
394 stations had issues.
Processing: MESH_CaSRv3p1/Average_GRU_Params/MESH_1p5p5_pr_forcast
Skipped 365 days from MESH_output_streamflow.csv
Metrics saved to metrics_MESH_CaSRv3p1_Average_GRU_Params_MESH_1p5p5_pr_forcast.csv
Metric descriptions saved to 'metrics_description.txt'
394 stations had issues.
Processing: MESH_CaSRv3p1/Distributed_GRU_Params/MESH_1p5p5
Skipped 365 days from MESH_output_streamflow